#### 08C · Calibración de f por mes (mapeo de cuantiles) y emulador completo
El PVAR no reproduce la cola alta de f (la parte que el XGB obtiene de las interacciones entre PCs). Aquí se corrige **directamente f**, el objeto que importa, sin tocar el generador:

1. f simulada: la que guardó el 08 en `emulador{SUF_IN}/sim_*.parquet` (horaria, XGB completo).
2. f de referencia: XGB completo con las **PCs históricas 1941–2025** (85 años).
3. Por mes: cuantil de cada valor simulado dentro de la f simulada de ese mes → mismo cuantil de la f histórica de ese mes. Conserva el orden (cronología y persistencia); por encima del último cuantil se conserva el exceso (no hay techo).
4. Ruido y vuelta a Hs exactamente como el 08 (mismas semillas) → `emulador{SUF_OUT}` + `am_pvar{SUF_OUT}.pkl` para el 09B.

**Versión rápida**: el ruido es el de siempre. Al quitar el déficit de f desaparece la compensación, así que es esperable que los niveles de retorno salgan algo altos (≈ +5 %, el sesgo conocido del ruido). Lo que se mira primero es la **frecuencia de temporales** y su **agrupamiento**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, json, glob, os, time
from xgboost import XGBRegressor
from scipy.stats import genextreme
from scipy.signal import find_peaks
from utils.pit.piece_wise_marginal import from_normal

In [2]:
#### Configuración
point = "p102"
SUF_IN = "_simple_est"                                      # simulaciones del PVAR ya pasadas por el 08 (las de producción)
SUF_RUIDO = "_simple_est"                                   # modelo de ruido (escala{SUF_RUIDO}.pkl)
SUF_OUT = SUF_IN + "_fqm"                                   # salida; en el 09B: SUF = SUF_OUT
N_SIMS = 20                                                # ← 20 para una primera lectura (≈ 18 s por simulación)
P_GRID = np.r_[np.linspace(0, 0.99, 991), 1 - np.logspace(-2, -5, 61)[1:]]   # cuantiles del mapeo (fino en la cola)

In [3]:
#### Transformación de Hs, XGB y ruido (idéntico al 08)
waves_df = pd.read_csv(f"inputs/waves_bulk_{point}.csv", index_col=0, parse_dates=True)
tr_ = pickle.load(open(f"outputs/marginal/transformacion_{point}.pkl", "rb"))
marg, harm_z = tr_["marg"], tr_["harm_z"]

def hs_desde_z(z):
    zz = pd.DataFrame({"VHM0": z, "VTM02": 0.0}, index=z.index)
    z0 = harm_z.inverse_transform(zz)["VHM0"].clip(-6.36, 6.36)
    return pd.Series(marg["VHM0"].ppf(from_normal(z0).values), index=z.index)

xgb_final = XGBRegressor(); xgb_final.load_model(f"outputs/xgb_pit_{point}/xgb_final.json")
feat = json.load(open(f"outputs/xgb_pit_{point}/xgb_features.json"))
cols, lags = feat["columns"], feat["lags_h"]

esc = pickle.load(open(f"outputs/xgb_pit_{point}/escala{SUF_RUIDO}.pkl", "rb"))
bordes, sigma, pools_r, results_u, L = esc["bordes"], esc["sigma"], esc["pools_r"], esc["results_u"], esc["L"]
SUAVIZADO_H = esc.get("suavizado_h", 1)
e_u = results_u.resid_["r"].asfreq("h"); ev = e_u.values
inicios = np.flatnonzero(np.convolve(~np.isnan(ev), np.ones(L), "valid") == L)

def ruido_bloques(n, L, rng):
    out = np.empty(n); t = 0
    while t < n:
        s = rng.choice(inicios); m = min(L, n - t); out[t:t + m] = ev[s:s + m]; t += m
    return out

def mapear(r, dec):
    out = np.empty(len(r))
    for k in pools_r:
        m = dec == k
        if m.sum():
            rk = (pd.Series(r.values[m]).rank().values - 0.5) / m.sum()
            out[m] = np.quantile(pools_r[k], rk)
    return pd.Series(out, index=r.index)

def f_desde_pcs(pcs):
    X = pd.concat({f"l{l}": pcs.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(pcs.index)
    X.columns = [f"{c_}_l{l[1:]}" for l, c_ in X.columns]
    X = X.dropna()[cols]
    f_h = pd.Series(xgb_final.predict(X), index=X.index).resample("h").interpolate()
    if SUAVIZADO_H > 1:
        f_h = f_h.rolling(SUAVIZADO_H, center=True, min_periods=1).mean()
    return f_h

def estacion(ix):
    return np.where(ix.month.isin([11, 12, 1, 2, 3]), 0, np.where(ix.month.isin([4, 10]), 1, 2))

def hs_desde_f(f_h, rng):
    dec = np.digitize(f_h, bordes)
    clase = dec + 10 * estacion(f_h.index)
    sig = pd.Series(sigma.loc[dec].values, index=f_h.index)
    ruido_df = pd.DataFrame(ruido_bloques(len(f_h), L, rng), index=f_h.index, columns=["r"])
    u = results_u.simulate(noise=ruido_df, burn=24 * 30)["r"]
    return hs_desde_z(f_h + mapear(u * sig, clase))

#### 1. f de referencia (PCs históricas 1941–2025) y f simulada (la guardada por el 08)

In [4]:
pcs_hist = pd.read_csv("outputs/pca/pcs_msl_sel_6h.csv", index_col=0, parse_dates=True)
f_ref = f_desde_pcs(pcs_hist.loc["1940-11-01":"2025-12-31"]).loc["1941":"2025"]

ficheros = sorted(glob.glob(f"outputs/xgb_pit_{point}/emulador{SUF_IN}/sim_*.parquet"))[:N_SIMS]
print(len(ficheros), "simulaciones")
f_sims = [pd.read_parquet(f_)["f"].astype("float64") for f_ in ficheros]

#### cuantiles por mes: simulados (todas las simulaciones juntas) e históricos
q_sim, q_ref = {}, {}
for m in range(1, 13):
    q_sim[m] = np.quantile(np.concatenate([f_[f_.index.month == m].values for f_ in f_sims]), P_GRID)
    q_ref[m] = np.quantile(f_ref[f_ref.index.month == m].values, P_GRID)

def calibra(f_):
    """Mapeo de cuantiles por mes; por encima del último cuantil se suma el exceso (sin techo)."""
    out = np.empty(len(f_)); v = f_.values; mes = f_.index.month
    for m in range(1, 13):
        sel = mes == m; x = v[sel]
        y = np.interp(x, q_sim[m], q_ref[m])
        alto, bajo = x > q_sim[m][-1], x < q_sim[m][0]
        y[alto] = q_ref[m][-1] + (x[alto] - q_sim[m][-1])
        y[bajo] = q_ref[m][0] + (x[bajo] - q_sim[m][0])
        out[sel] = y
    return pd.Series(out, index=f_.index)

#### comprobación de la cola por mes (q99 sim/ref) antes y después
q99 = pd.DataFrame({"antes": [np.quantile(np.concatenate([f_[f_.index.month == m] for f_ in f_sims[:10]]), 0.99) for m in range(1, 13)],
                    "después": [np.quantile(np.concatenate([calibra(f_)[f_.index.month == m] for f_ in f_sims[:10]]), 0.99) for m in range(1, 13)]},
                   index=range(1, 13)).div(f_ref.groupby(f_ref.index.month).quantile(0.99).values, axis=0)
print("q99 de f, sim/hist por mes:"); display(q99.T.round(2))

20 simulaciones
q99 de f, sim/hist por mes:


,1,2,3,4,5,6,7,8,9,10,11,12
antes,0.91,0.93,0.87,0.84,0.80,0.92,0.82,0.79,0.81,0.84,0.95,0.92
después,0.99,1.00,1.01,1.01,0.99,1.00,0.99,0.98,0.98,0.99,1.01,0.99


#### 2. Emulador con f calibrada: ruido y Hs como el 08 (mismas semillas)

In [5]:
carpeta = f"outputs/xgb_pit_{point}/emulador{SUF_OUT}"
os.makedirs(carpeta, exist_ok=True)
u_hist = waves_df.loc["1985":"2025", "VHM0"].quantile(0.99)
am_hs, am_f, stats, q_f, dur = [], [], [], [], []
for i, (fic, f_h) in enumerate(zip(ficheros, f_sims)):
    t = time.time()
    f_c = calibra(f_h)
    hs = hs_desde_f(f_c, np.random.default_rng(i))
    pd.DataFrame({"VHM0": hs, "f": f_c}).astype("float32").to_parquet(f"{carpeta}/sim_{i:03d}.parquet")
    am_hs.append(hs.groupby(hs.index.year).max().reset_index(drop=True))
    am_f.append(f_c.groupby(f_c.index.year).max().reset_index(drop=True))
    q_f.append(f_c.quantile([0.5, 0.9, 0.99, 0.999]))
    n_anos = hs.index.year.nunique()
    pk = find_peaks(hs.values, height=u_hist, distance=72)[0]
    stats.append({"q99": hs.quantile(0.99), "q999": hs.quantile(0.999), "max": hs.max(),
                  "temporales/10 años": len(pk) * 10 / n_anos})
    #### duración media de las rachas por encima del umbral (agrupamiento)
    sobre = (hs > u_hist).astype(int).values
    cambios = np.diff(np.r_[0, sobre, 0]); dur.append(np.mean(np.flatnonzero(cambios == -1) - np.flatnonzero(cambios == 1)))
    print(f"sim {i:03d} ({time.time() - t:.0f} s)", flush=True)
am_hs = pd.concat(am_hs, axis=1); am_f = pd.concat(am_f, axis=1); stats = pd.DataFrame(stats)
stats["duración media > q99 (h)"] = dur
pickle.dump({"am_hs": am_hs, "am_f": am_f, "stats": stats}, open(f"outputs/xgb_pit_{point}/am_pvar{SUF_OUT}.pkl", "wb"))

sim 000 (17 s)
sim 001 (17 s)
sim 002 (17 s)
sim 003 (17 s)
sim 004 (17 s)
sim 005 (17 s)
sim 006 (17 s)
sim 007 (17 s)
sim 008 (17 s)
sim 009 (17 s)
sim 010 (17 s)
sim 011 (17 s)
sim 012 (17 s)
sim 013 (17 s)
sim 014 (17 s)
sim 015 (17 s)
sim 016 (17 s)
sim 017 (17 s)
sim 018 (17 s)
sim 019 (17 s)


#### 3. Resultados: f, temporales, agrupamiento y niveles de retorno

In [6]:
#### f
f_hist85 = f_ref
tabla_f = pd.DataFrame({"f histórica 1941–2025": f_hist85.quantile([0.5, 0.9, 0.99, 0.999]),
                        "f calibrada med": pd.concat(q_f, axis=1).median(axis=1)})
print(tabla_f.round(3).to_string())
print(f"máx. anual de f: histórica {f_hist85.groupby(f_hist85.index.year).max().mean():.2f} | calibrada {am_f.values.mean():.2f}")

#### Hs y temporales (referencia sin calibrar, 08: 43,9 temporales/década)
hs_obs = waves_df.loc["1985":"2025", "VHM0"]
pk_o = find_peaks(hs_obs.values, height=u_hist, distance=72)[0]
so = (hs_obs > u_hist).astype(int).values; co = np.diff(np.r_[0, so, 0])
print(f"\nobs : q99 {hs_obs.quantile(0.99):.2f} | q999 {hs_obs.quantile(0.999):.2f} | max {hs_obs.max():.2f} | "
      f"temporales/10 años {len(pk_o) * 10 / hs_obs.index.year.nunique():.0f} | duración media > q99 {np.mean(np.flatnonzero(co == -1) - np.flatnonzero(co == 1)):.1f} h")
print(stats.quantile([0.025, 0.5, 0.975]).round(2).to_string())

#### GEV en bloques de 41 años (como el 08)
am_obs = hs_obs.groupby(hs_obs.index.year).max(); n_b = len(am_obs)
def gev(am):
    c_, loc, scale = genextreme.fit(am)
    return {"ξ": -c_, **{f"Hs{T}": genextreme.ppf(1 - 1 / T, c_, loc, scale) for T in [2, 10, 50, 100]}}
bloques = [am_hs[c_].values[j:j + n_b] for c_ in am_hs.columns for j in range(0, len(am_hs) - n_b + 1, n_b)]
g_b = pd.DataFrame([gev(b) for b in bloques])
print(pd.DataFrame({"obs": pd.Series(gev(am_obs.values)), "calibrada med": g_b.median(),
                    "lo": g_b.quantile(0.025), "hi": g_b.quantile(0.975)}).round(2).to_string())
print(f"\nEn el 09B: SUF = '{SUF_OUT}'")

       f histórica 1941–2025  f calibrada med
0.500                 -0.032           -0.032
0.900                  0.958            0.959
0.990                  1.925            1.922
0.999                  2.667            2.687
máx. anual de f: histórica 2.67 | calibrada 2.65

obs : q99 3.73 | q999 5.30 | max 7.78 | temporales/10 años 62 | duración media > q99 10.4 h
        q99  q999   max  temporales/10 años  duración media > q99 (h)
0.025  3.79  5.54  7.81               56.26                     12.18
0.500  3.90  5.79  8.62               59.48                     13.14
0.975  4.01  5.98  9.82               64.90                     13.88
        obs  calibrada med    lo    hi
ξ     -0.13          -0.13 -0.33  0.11
Hs2    5.59           5.80  5.46  6.07
Hs10   6.67           7.14  6.67  7.59
Hs50   7.43           8.11  7.40  9.09
Hs100  7.70           8.44  7.57  9.93

En el 09B: SUF = '_simple_est_fqm'


#### Cómo leerlo
- **Temporales/década**: referencia sin calibrar 43,9 (obs 62). Si se acerca a 62, la frecuencia se arregla calibrando f.
- **Duración media > q99** y, en el 09B (sección 5B), θ e intervalos entre temporales: comprueban que el agrupamiento no se estropea (el aviso de que el mapeo corrige la marginal de f, no necesariamente cómo se agrupan los extremos).
- **GEV**: es esperable que Hs10–Hs100 salgan algo altos (≈ +5 %): es el sesgo del ruido que antes tapaba el PVAR. Si sale así, el siguiente paso es la cola del ruido (intervalos finos) en modo histórico.
- Fronteras de mes: el mapeo cambia de un mes a otro de golpe; si se ven saltos el día 1, suavizar con una ventana móvil de ±15 días.